# Universidad de Buenos Aires
# Aprendizaje Profundo - TP3
# Cohorte 24 - 2do bimestre 2026


Este tercer y último TP se debe entregar hasta las **23:59hs del viernes 19 de junio (hora de Argentina)**. La resolución del TP es **individual**. Pueden utilizar los contenidos vistos en clase y otra bibliografía que consideren que les haga falta. Si se toman ideas de fuentes externas deben ser correctamente citadas incluyendo el correspondiente link o página de libro.

ESTE TP3 EQUIVALE A UN TERCIO DE SU NOTA FINAL.

El formato de entrega debe ser un link a un notebook de google colab. Permitir acceso a gvilcamiza.ext@fi.uba.ar y **habilitar los comentarios, para poder darles el feedback**. Si no lo hacen así no se podrá dar el feedback respectivo por cada pregunta.

El envío **se realizará en el siguiente link de google forms: [link](https://forms.gle/56R6couXZBPDZzfs5)**. Tanto los resultados, gráficas, como el código y las explicaciones deben quedar guardados y visualizables en el colab.

**NO SE VALIDARÁN ENVÍOS POR CORREO, EL MÉTODO DE ENTREGA ES SOLO POR EL FORMS.**

**Consideraciones a tener en cuenta:**
- Se entregará 1 solo colab para este TP3.
- Renombrar el archivo de la siguiente manera: **APELLIDO-NOMBRE-DL-TP3-Co24.ipynb**
- Los códigos deben poder ejecutarse.
- Los resultados, cómo el código, los gráficos y las explicaciones deben quedar guardados y visualizables en el correspondiente notebook.
- Prestar atención a las consignas, responder las preguntas cuando corresponda.
- Solo se revisarán los trabajos que hayan sido enviados por el forms.

# **CLASIFICADOR DE CANCIONES SEGÚN GENEROS MUSICALES**

El objetivo de este trabajo es construir una red neuronal recurrente (RNN) utilizando Pytorch, capaz de clasificar géneros musicales a partir del fragmento de una canción en formato .wav. El clasificador deberá identificar uno de los 10 géneros : blues, classical, country, disco, hiphop, jazz, metal, pop, reggae o rock. El dataset se encuentra en este link: https://drive.google.com/file/d/1txyVxGVezQ4fcHqFi-n3gcWKEYQexu1_/view?usp=sharing

# 1. Preprocesamiento de los datos (2 puntos)

Antes de entrenar el modelo de clasificación, es necesario transformar los archivos de audio en una representación numérica que pueda ser procesada por una red neuronal recurrente (RNN). En este trabajo se utilizarán **MFCCs (Mel-Frequency Cepstral Coefficients)**, una de las representaciones más empleadas en tareas de análisis de audio, reconocimiento de voz y clasificación musical.

El flujo general de preprocesamiento es el siguiente:

```text
Archivo WAV
    ↓
Carga del audio
    ↓
Conversión a una frecuencia de muestreo común
    ↓
Extracción de MFCCs
    ↓
Ajuste de longitud temporal
    ↓
Normalización
    ↓
Conversión a tensor
```

## A. Carga del audio

Cada archivo de audio es cargado utilizando la función **`librosa.load()`** de la biblioteca **Librosa**. Durante este proceso se establece una frecuencia de muestreo uniforme para todos los archivos.

La frecuencia de muestreo utilizada puede ser:

$$
f_s = 22050 Hz
$$

Esto significa que cada segundo de audio se representa mediante 22 050 muestras. **Pero pueden elegir otra fs si lo consideran mejor.**

Además, se considera una duración máxima de 30 segundos por canción, correspondiente a la duración estándar de los audios del dataset GTZAN.

## B. Extracción de características mediante MFCC

Una vez cargado el audio, se calculan los **Mel-Frequency Cepstral Coefficients (MFCC)** mediante la función **`librosa.feature.mfcc()`**.

Los MFCC permiten representar el contenido espectral del audio de una manera compacta, capturando características relevantes para distinguir entre distintos géneros musicales.

Un valor balanceado podría ser:

$$
n_{MFCC} = 40
$$

coeficientes por cada instante temporal. **Pero pueden cambiarlo si desean.**

Conceptualmente, los MFCC se obtienen mediante las siguientes etapas:

1. Transformada rápida de Fourier (FFT).
2. Aplicación de filtros en escala Mel.
3. Obtención de la energía logarítmica de cada banda.
4. Transformada discreta del coseno (DCT).

La conversión entre frecuencia real y frecuencia Mel se define como:

$$
Mel(f)=2595\log_{10}\left(1+\frac{f}{700}\right)
$$

Esta escala aproxima la forma en que el oído humano percibe las diferencias de frecuencia.

Tras la extracción de características, cada canción queda representada por una matriz de dimensiones:

**(40, T)**

donde:

* **40** representa la cantidad de coeficientes MFCC calculados para cada frame.
* **T** representa el número de frames temporales.

Pero recomiendo hallar la transpuesta `.T` para obtener **(T, 40)**.

## C. Ajuste de longitud temporal

Las RNNs requieren que todas las muestras de un mismo lote tengan dimensiones consistentes.

Sin embargo, debido a pequeñas diferencias en la duración efectiva de los audios y en el proceso de extracción de características, la cantidad de frames temporales puede variar ligeramente entre canciones.

Para solucionar este problema se define una longitud máxima, la cual puede ser:

$$
T_{max}=1300
$$

Si una secuencia contiene menos de 1300 frames, se agregan valores nulos al final de la secuencia (*padding*).

Si una secuencia contiene más de 1300 frames, se eliminan los frames excedentes (*truncation*).

De esta forma, todas las canciones quedan representadas mediante matrices de tamaño:

**(1300, 40)**

lo que facilita la construcción de lotes de entrenamiento.

**Pero no es obligatorio que usen 1300, prueben con otro valor a ver si mejora.**

## D. Normalización de características

Antes de alimentar los datos al modelo, los MFCC se normalizan para reducir diferencias de escala entre muestras.

La normalización utilizada corresponde a una estandarización tipo Z-score:

$$
x_{norm}=\frac{x-\mu}{\sigma}
$$

donde:

* $x$ es el valor original.
* $\mu$ es la media de la muestra.
* $\sigma$ es la desviación estándar.

Tras este proceso, los datos presentan aproximadamente:

* Media cercana a cero.
* Desviación estándar cercana a uno.

La normalización mejora la estabilidad numérica y acelera la convergencia durante el entrenamiento.

## E. Conversión a tensores

Finalmente, las matrices obtenidas se convierten a tensores de **PyTorch** mediante la función **`torch.tensor()`**.

Cada muestra queda representada por una secuencia temporal con dimensiones:

**(1300, 40)**

donde:

* 1300 corresponde al número de pasos temporales.
* 40 corresponde al número de coeficientes MFCC por cada paso temporal.

Durante el entrenamiento, las muestras se agrupan en lotes (*batches*), por lo que la entrada al modelo tendrá la forma:

**(BatchSize, 1300, 40)**

donde:

* **BatchSize** representa la cantidad de canciones procesadas simultáneamente.
* **1300** representa la longitud temporal de la secuencia.
* **40** representa el número de características extraídas por instante temporal.

Estas secuencias constituyen la entrada del modelo recurrente encargado de clasificar la canción en uno de los diez géneros musicales presentes en el dataset GTZAN.


In [6]:
from pathlib import Path
import numpy as np
import librosa
import torch

In [8]:
DATASET_PATH = "datos_music/genres_original"

SR = 22050
N_MFCC = 40
MAX_LEN = 1300

data_dir = Path(DATASET_PATH)
genres = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])
label_to_idx = {genre: i for i, genre in enumerate(genres)}
idx_to_label = {i: genre for genre, i in label_to_idx.items()}

def extraer_mfcc(ruta_audio, sr=SR, n_mfcc=N_MFCC, max_len=MAX_LEN):
  y, _ = librosa.load(ruta_audio, sr=sr, mono=True)
  mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T  # (T, 40)

  if mfcc.shape[0] < max_len:
    mfcc = np.pad(mfcc, ((0, max_len - mfcc.shape[0]), (0, 0)), mode="constant")
  else:
    mfcc = mfcc[:max_len, :]

  mean = mfcc.mean(axis=0, keepdims=True)
  std = mfcc.std(axis=0, keepdims=True)
  std[std == 0] = 1.0
  mfcc = (mfcc - mean) / std

  return mfcc.astype(np.float32)

X, y, archivos = [], [], []

for ruta in sorted(data_dir.glob("*/*.wav")):
  try:
    X.append(extraer_mfcc(ruta))
    y.append(label_to_idx[ruta.parent.name])
    archivos.append(str(ruta))
  except Exception as e:
    print(f"Error procesando {ruta}: {e}")

X = np.stack(X)  # (N, 1300, 40)
y = np.array(y, dtype=np.int64)

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
dataset_tensor = torch.utils.data.TensorDataset(X_tensor, y_tensor)

print("Géneros:", genres)
print("Cantidad de archivos procesados:", len(archivos))
print("Shape X:", X_tensor.shape)
print("Shape y:", y_tensor.shape)

Géneros: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Cantidad de archivos procesados: 999
Shape X: torch.Size([999, 1300, 40])
Shape y: torch.Size([999])


## 2. Construcción y entrenamiento del modelo recurrente (4 puntos)

* Construir dos modelos de clasificación de géneros musicales utilizando PyTorch. Uno basado en una RNN clásica y el otro en una LSTM.
* No se deben utilizar modelos pre-entrenados. La arquitectura debe ser implementada y entrenada desde cero.
* Analizar correctamente qué funciones de activación son adecuadas, qué tamaño de estado oculto se empleará, cuántas capas recurrentes tendrán los modelos, si se usará dropout, qué learning rate se aplicará, qué función de costo se utilizará y qué optimizador se empleará.


## 3. Evaluación del Modelo (2.5 puntos)

Cada modelo entrenado debe ser evaluado utilizando las siguientes métricas:

- **Accuracy**:
  - Reportar el valor final en el conjunto de validación.
  - Incluir una gráfica de evolución por época para entrenamiento y validación.

- **F1 Score Macro**:
  - Reportar el valor final en el conjunto de validación.
  - Incluir una gráfica de evolución por época para entrenamiento y validación.

- **Costo (Loss)**:
  - Mostrar una gráfica de evolución del costo por época para entrenamiento y validación.

- **Classification report**
  - Mostrar la precisión, recall y F1 score por cada clase usando `classification_report`

- **Matriz de confusión**:
  - Mostrar la matriz de confusión absoluta (valores enteros).
  - Mostrar la matriz de confusión normalizada (valores entre 0 y 1 por fila).

- **Comparativa de modelos**:
  - Mostrar una tabla comparativa de ambos modelos y explicar cuál es que ustedes consideran como modelo ganador.

Se recomienda utilizar `scikit-learn` para calcular métricas como accuracy, F1 score, el Classification report y las matrices de confusión. Las visualizaciones pueden realizarse con `matplotlib` o `seaborn`.


## 4. Prueba con canciones nuevas (1.5 punto)

Seleccionar al menos **10 canciones que no formen parte del dataset GTZAN**, las cuales serán utilizadas para evaluar la capacidad de generalización del modelo ganador.

Las canciones pueden provenir de plataformas de música, bibliotecas de audio libres de derechos, colecciones personales o descargadas de youtube. No está permitido utilizar canciones pertenecientes al dataset de entrenamiento, validación o prueba utilizado durante el desarrollo del modelo.

* Debe haber al menos una canción representativa de cada uno de los géneros musicales presentes en GTZAN.

* Aplicar exactamente el mismo proceso de preprocesamiento utilizado durante el entrenamiento del modelo, incluyendo:

  * Conversión a la frecuencia de muestreo utilizada.
  * Extracción de MFCCs.
  * Ajuste de longitud temporal mediante padding o truncation.
  * Normalización de las características.

* Pasar cada canción por el **modelo ganador** entrenado y mostrar:

  * Nombre de la canción.
  * Score asignado a cada género musical (normalizado entre 0 y 1 o entre 0% y 100%).
  * Género musical esperado.
  * La clase ganadora inferida por el modelo.
  * Un reproductor dentro del notebook hecho con `from IPython.display import Audio, display` por cada canción.

* Analizar los casos en los que el modelo se equivoca e intentar identificar posibles causas, tales como:

  * Similitudes entre géneros musicales.
  * Calidad del audio.
  * Mezcla de estilos musicales.
  * Limitaciones del dataset utilizado para el entrenamiento.

* Redactar conclusiones sobre la capacidad de generalización del modelo frente a canciones que nunca fueron vistas durante el entrenamiento.
